# 8. Aggregation: cells that follow their own signal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/tutorials/08_aggregation.ipynb)

Starving *Dictyostelium* amoebae secrete cAMP and move up its gradient;
immune cells and bacteria do the same with their own chemokines. Where a
few more cells gather by chance, they secrete more, the signal rises, and
it draws in their neighbours, which secrete in turn. This positive feedback
makes a uniform population unstable: the cells aggregate. Keller and Segel
wrote down the continuum model in 1970.

**Learning objectives**

- couple a field that the cells secrete to their movement;
- find the chemotactic strength at which aggregation sets in;
- see how the decay of the signal sets the spacing of aggregates; and
- read a kymograph of the cells and of the field.

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca.fields import PDESpec
from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import (
    InteractionPipelineSpec,
    ReorientationSpec,
    ReorientationTermSpec,
)
from lgca.simulation import DensityRecorder, FieldRecorder, Schedule
from lgca.study import sweep

## The model

The chemokine $c$ diffuses, decays at rate $k$ and is secreted at rate
$\alpha$ by each of the $n$ cells at a node:

$$\partial_t c = D\,\Delta c + \alpha\, n - k\, c.$$

The cells reorient with the `chemotaxis` term (tutorial 3): a channel state
whose flux $J$ points up the gradient is more likely, $P \propto
\exp(\beta\, \nabla c \cdot J)$, with the chemotactic strength $\beta$.

A time step is the `pde` operator, which updates the chemokine with the
cells where they are, then the reorientation, then propagation. The
chemokine diffuses about as fast as the cells move ($D = 1$ node² per step),
so it is not at a steady state; the default solver, `"implicit"`, advances
it by one step. The cells are not limited to one per channel
(`volume_exclusion=False`), so aggregates can hold many cells.

In [ ]:
def aggregation_spec(beta, decay, length=200, steps=400, seed=1, every=2):
    return ModelSpec(
        description=Description(title=f"Aggregation, beta = {beta}, decay = {decay}"),
        space=SpaceSpec(geometry="lin", dims=length, boundary="periodic"),
        state=StateSpec(
            density=3.0,
            restchannels=1,
            volume_exclusion=False,
            capacity=50,
            fields={"chemokine": 0.0},
        ),
        time=TimeSpec(steps=steps, seed=seed),
        dynamics=InteractionPipelineSpec(
            operators=[
                PDESpec(field="chemokine", diffusion=1.0, decay=decay, cells=[{"production": 0.1}]),
                ReorientationSpec(terms=[
                    ReorientationTermSpec(name="chemotaxis", beta=beta, parameters={"field": "chemokine"}),
                ]),
            ],
        ),
        analysis=AnalysisSpec(
            observers=[
                DensityRecorder(schedule=Schedule(every=every)),
                FieldRecorder(["chemokine"], schedule=Schedule(every=every)),
            ],
        ),
    )


run = run_model(aggregation_spec(beta=0.3, decay=0.01), showprogress=False)

A kymograph shows a 1D model over time: the lattice runs along the x
axis, time down the y axis. `plot_scalarfield` draws one from the recorded
field; the cells we draw the same way with `imshow`:

In [ ]:
steps = run.data.steps("density")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
image = axes[0].imshow(run.data["density"], aspect="auto", cmap="hot_r",
                       extent=(0, 200, steps[-1], steps[0]), vmax=30)
fig.colorbar(image, ax=axes[0], label="cells per node")
axes[0].set(xlabel="node", ylabel="time step", title="cells")
run.lgca.plot_scalarfield(run.data["chemokine"], steps=steps, ax=axes[1], cbarlabel="chemokine")
axes[1].set(title="chemokine")
plt.show()
plt.close(fig)

Within about a hundred steps the random initial state breaks up into
aggregates, each sitting on a peak of chemokine that it makes itself.
Neighbouring aggregates then merge: the larger one secretes more and pulls
the smaller one in.

## When does aggregation set in?

Chemotaxis must beat the random motion of the cells. For the continuum
Keller–Segel model, a small ripple of wavelength $2\pi/q$ in a uniform
population of $n_0$ cells grows if

$$\chi\, \alpha\, n_0 > D_n\, (D\, q^2 + k),$$

where $\chi$ is the chemotactic sensitivity (here it grows with $\beta$) and
$D_n$ the diffusion coefficient of the cells. Two predictions follow: there
is a threshold of chemotactic strength, and decay raises it. Without decay
the longest ripple that fits on the lattice sets the threshold, which is
then low.

A sweep over $\beta$ and the decay, with three seeds each, measures how
clustered the cells are at the end: the coefficient of variation of the
cells per node, which is about $1/\sqrt{n_0} \approx 0.6$ for cells placed
at random.

In [ ]:
def clustering(result):
    """Standard deviation over mean of the cells per node, at the end of the run."""
    density = result.lgca.cell_density[result.lgca.nonborder]
    return density.std() / density.mean()


onset = sweep(
    aggregation_spec(beta=0.0, decay=0.0, steps=500),
    grid={"beta": [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4], "decay": [0.0, 0.01, 0.1]},
    seeds=range(3),
    measure={"clustering": clustering},
    showprogress=False,
)
mean = onset.groupby(["decay", "beta"])["clustering"].mean()

fig, axis = plt.subplots(figsize=(6, 3.8), constrained_layout=True)
for decay in (0.0, 0.01, 0.1):
    axis.plot(mean[decay].index, mean[decay].values, marker="o", label=f"decay {decay}")
axis.axhline(1 / np.sqrt(3.0), color="grey", linestyle=":", label="random placement")
axis.set(xlabel="chemotactic strength beta", ylabel="clustering (std / mean)")
axis.legend()
plt.show()
plt.close(fig)

Below a threshold the cells stay as clustered as random placement; above
it they aggregate. The threshold rises with the decay, as the continuum
theory predicts: a signal that decays is weaker, so the cells must follow it
more strongly.

## Decay sets the spacing of aggregates

A decaying signal falls off with the distance from where it is made, over
the decay length $\sqrt{D / k}$: 3 nodes for $k = 0.1$, 10 for $k = 0.01$.
Aggregates many decay lengths apart hardly feel each other, so merging
slows to a halt. Without decay, the signal of every aggregate spreads over
the whole lattice, and merging goes on. Longer runs on a longer lattice show
the difference:

In [ ]:
def aggregates(density, level=10.0):
    """The number of runs of neighbouring nodes with more than level cells (periodic)."""
    high = density > level
    return int(np.sum(high & ~np.roll(high, 1)))


long_runs = {decay: run_model(aggregation_spec(beta=0.3, decay=decay, length=400, steps=3000, every=20),
                              showprogress=False)
             for decay in (0.1, 0.0)}

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for axis, (decay, result) in zip(axes, long_runs.items()):
    steps = result.data.steps("density")
    axis.imshow(result.data["density"], aspect="auto", cmap="hot_r",
                extent=(0, 400, steps[-1], steps[0]), vmax=30)
    axis.set(xlabel="node", ylabel="time step", title=f"cells, decay {decay}")
    counts = [aggregates(density) for density in result.data["density"]]
    axes[2].plot(steps, counts, label=f"decay {decay}")
axes[2].set(xlabel="time step", ylabel="aggregates")
axes[2].legend()
plt.show()
plt.close(fig)

With decay, the number of aggregates settles; without decay it keeps
falling as aggregates merge, a process called coarsening.

## Aggregation in two dimensions

The same model on a square lattice, with stronger chemotaxis. In two
dimensions the aggregates of the continuum model can collapse to points (a
finite-time blow-up); on the lattice they shrink to a node or two, each
holding hundreds of cells.

In [ ]:
def aggregation_2d(decay, beta=2.0, size=80, steps=600):
    return ModelSpec(
        description=Description(title=f"Aggregation on a square lattice, decay = {decay}"),
        space=SpaceSpec(geometry="square", dims=(size, size), boundary="periodic"),
        state=StateSpec(density=1.0, restchannels=1, volume_exclusion=False, capacity=50,
                        fields={"chemokine": 0.0}),
        time=TimeSpec(steps=steps, seed=2),
        dynamics=InteractionPipelineSpec(
            operators=[
                PDESpec(field="chemokine", diffusion=1.0, decay=decay, cells=[{"production": 0.1}]),
                ReorientationSpec(terms=[
                    ReorientationTermSpec(name="chemotaxis", beta=beta, parameters={"field": "chemokine"}),
                ]),
            ],
        ),
        analysis=AnalysisSpec(
            observers=[
                DensityRecorder(schedule=Schedule(every=100)),
                FieldRecorder(["chemokine"], schedule=Schedule(every=100)),
            ],
        ),
    )


runs_2d = {decay: run_model(aggregation_2d(decay), showprogress=False) for decay in (0.1, 0.0)}

fig, axes = plt.subplots(2, 3, figsize=(11, 7.4), constrained_layout=True)
for row, (decay, result) in enumerate(runs_2d.items()):
    for column, index in enumerate((0, 1, -1)):
        step = result.data.steps("density")[index]
        axes[row, column].imshow(result.data["density"][index].T, origin="lower", cmap="hot_r", vmax=20)
        axes[row, column].set(title=f"decay {decay}, step {step}", xticks=[], yticks=[])
plt.show()
plt.close(fig)

With decay, more aggregates remain; without decay, more of them have
merged into fewer, larger ones. `FieldRecorder` recorded the
chemokine as well; `result.lgca.animate_scalarfield(result.data["chemokine"],
steps=result.data.steps("chemokine"))` animates it.

## Exercises

1. Change the diffusion coefficient of the chemokine to 0.25 and to 4, with
   decay 0.01. How does the number of aggregates change?
2. Turn volume exclusion on (`volume_exclusion=True`, no capacity). How
   large can an aggregate become, and what happens to the threshold?
3. The chemokine of real cells may be fast compared with their movement.
   Use `solver="steady"` with decay 0.1 (a steady field needs something
   that removes it). Does the threshold change?
4. Add growth: `{"name": "birth_death", "parameters": {"birth_rate": 0.01,
   "death_rate": 0.01}}` before the `pde` operator. Do the aggregates
   persist?
5. Measure the number of aggregates at the end of the long run for several
   decay rates. Is their spacing proportional to the decay length
   $\sqrt{D/k}$?